# Notebook 1: Getting started with QSARmil

This notebook is for anyone who just wants to build a model and get predictions, without needing to know all details how the
pipeline works under the hood. You give QSARmil a list of molecules (as SMILES strings) and a list of their property values
(regression or binary classification), and QSARmil takes care of everything else: generating conformers,
calculating 3D descriptors, training several models, and selecting the optimal consensus of models.

In [7]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "" # uncomment if you do not have GPU

In [2]:
import pandas as pd
from sklearn.metrics import r2_score

from qsarmil import MultiConformerRegressor, MultiConformerClassifier

### 1. Load your data

QSARmil expects two things: a list of molecules written as SMILES strings, and a list of the property values you
want to predict (one number per molecule).

As an example, we use a small, publicly available dataset of binding activities, from:

> Van Tilborg, Derek, Alisa Alenicheva, and Francesca Grisoni. "Exposing the limitations of molecular machine
> learning with activity cliffs." Journal of chemical information and modeling 62.23 (2022): 5938-5951.

In [3]:
url = "https://raw.githubusercontent.com/molML/MoleculeACE/main/MoleculeACE/Data/benchmark_data/CHEMBL2034_Ki.csv"
df_ace = pd.read_csv(url)

df_train = df_ace[df_ace["split"] == "train"][["smiles", "y"]].reset_index(drop=True)
df_test = df_ace[df_ace["split"] == "test"][["smiles", "y"]].reset_index(drop=True)

df_train.shape, df_test.shape

((598, 2), (152, 2))

<div class="alert alert-block alert-info">
<b>💡 Tip: </b>Training a full model on the whole dataset can take a while, since QSARmil tries many different model
combinations. If you just want to try things out quickly (for example, to check that everything runs on your
machine), keep the cell below uncommented to work with a small random sample instead. Comment it out once
you're ready to run the real thing.
</div>

In [4]:
# uncomment to work with a small sample instead of the full dataset (much faster, good for a first try)
df_train = df_train.sample(n=15, random_state=42).reset_index(drop=True)
df_test = df_test.sample(n=5, random_state=42).reset_index(drop=True)

df_train.shape, df_test.shape

((15, 2), (5, 2))

### 2. Build a multi-conformer model

Behind the scenes, QSARmil default pipeline does the following for you:

1. **Generate conformers:** a number of conformers are generated for each molecule (with `RDKit`)
2. **Calculate 3D descriptors:** each conformer is encoded with several 3D descriptors (9 descriptors from `RDKit` and `MolFeat`)
3. **Apply multi-instance learning methods:** several different MIL methods are applied (8 methods from `milearn`), one for each descriptor type
4. **Genetic consensus optimization:** finally, a genetic algorithm is applied to find the best consensus of models (usually 5-10 models)

Two learning tasks are currently available:

- **`MultiConformerRegressor`** - for regression tasks
- **`MultiConformerClassifier`** - for binary classification tasks

Both classes accept the same parameters for the modelling pipeline:

- `num_conf` - how many conformers to generate per molecule
- `hopt` - whether QSARmil should optimize MIL method hyperparameters (`"True"` or `"False"`)
- `num_cpu` - how many CPUs to use in the pipeline
- `output_folder` - where QSARmil writes its files. Default is a timestamped folder created automatically, e.g. `qsarmil_27_08_2026_23_09_47`
- `verbose` - whether to print progress while training (`"True"` or `"False"`)
- `random_seed` - a fixed random number so that re-running the same code gives the same result
- `accelerator` - whether to train on `"cpu"` or `"gpu"`

There's only one method you need, and it does everything in one call - training, consensus selection, and prediction:

- **`y_test_pred = model.train_predict(smiles_train, y_train, smiles_test)`** - trains the model on your data and returns
  predictions for `smiles_test`

In [5]:
smiles_train, y_train = df_train["smiles"].to_list(), df_train["y"].to_list()
smiles_test = df_test["smiles"].to_list()

model = MultiConformerRegressor(
    num_conf=10,          # generate up to 10 conformers per molecule
    hopt=False,            # use False for speed
    num_cpu=4,              # use 4 CPU threads for conformer generation
    output_folder="mcr-1",   # if None, QSARmil creates a timestamped folder automatically
    verbose=True,           # print progress while training
    random_seed=42,         # fixed random seed, for reproducible results
    accelerator="cpu",      # train on CPU ("gpu" is also available if you have one)
)
y_test_pred = model.train_predict(smiles_train, y_train, smiles_test)

Step-1. Conformer generation
Generated conformers for 20 of 20 molecules
Step-2. Descriptor calculation
RDKitGEOM: done
RDKitAUTOCORR: done
RDKitRDF: done
RDKitMORSE: done
RDKitWHIM: done
MolFeatUSRD: done
MolFeatElectroShape: done
RDKitGETAWAY: done
MolFeatPmapper: done
Step-3. Individual model training
[1/72] RDKitGEOM|MeanInstanceWrapperMLPNetworkRegressor
[2/72] RDKitGEOM|MeanBagWrapperMLPNetworkRegressor
[3/72] RDKitGEOM|MeanBagNetworkRegressor
[4/72] RDKitGEOM|MeanInstanceNetworkRegressor
[5/72] RDKitGEOM|AdditiveAttentionNetworkRegressor
[6/72] RDKitGEOM|SelfAttentionNetworkRegressor
[7/72] RDKitGEOM|HopfieldAttentionNetworkRegressor
[8/72] RDKitGEOM|DynamicPoolingNetworkRegressor
[9/72] RDKitAUTOCORR|MeanInstanceWrapperMLPNetworkRegressor
[10/72] RDKitAUTOCORR|MeanBagWrapperMLPNetworkRegressor
[11/72] RDKitAUTOCORR|MeanBagNetworkRegressor
[12/72] RDKitAUTOCORR|MeanInstanceNetworkRegressor
[13/72] RDKitAUTOCORR|AdditiveAttentionNetworkRegressor
[14/72] RDKitAUTOCORR|SelfAttentio

### 3. Acces individual model predictions

Genetic consensus search is already included in the pipeline, but if you want to acces the predictions for all individual models, you can just read the source file in the result folder:

In [6]:
df_pred = pd.read_csv("mcr-1/test.csv")
df_pred

,SMILES,RDKitGEOM|MeanInstanceWrapperMLPNetworkRegressor,RDKitGEOM|MeanBagWrapperMLPNetworkRegressor,RDKitGEOM|MeanBagNetworkRegressor,RDKitGEOM|MeanInstanceNetworkRegressor,RDKitGEOM|AdditiveAttentionNetworkRegressor,RDKitGEOM|SelfAttentionNetworkRegressor,RDKitGEOM|HopfieldAttentionNetworkRegressor,RDKitGEOM|DynamicPoolingNetworkRegressor,RDKitAUTOCORR|MeanInstanceWrapperMLPNetworkRegressor,...,RDKitGETAWAY|HopfieldAttentionNetworkRegressor,RDKitGETAWAY|DynamicPoolingNetworkRegressor,MolFeatPmapper|MeanInstanceWrapperMLPNetworkRegressor,MolFeatPmapper|MeanBagWrapperMLPNetworkRegressor,MolFeatPmapper|MeanBagNetworkRegressor,MolFeatPmapper|MeanInstanceNetworkRegressor,MolFeatPmapper|AdditiveAttentionNetworkRegressor,MolFeatPmapper|SelfAttentionNetworkRegressor,MolFeatPmapper|HopfieldAttentionNetworkRegressor,MolFeatPmapper|DynamicPoolingNetworkRegressor
0,C[C@H]1c2c(cc(F)c(-c3cccc4c(Cl)c[nH]c34)c2F)NC...,-1.046218,-1.565738,-2.426204,-2.426204,-2.515816,-2.155852,-1.679124,-1.963517,-1.455104,...,-0.942751,-1.314872,-0.268802,-0.322267,-0.420099,-0.420101,-0.379518,-0.708642,0.004910,-0.449618
1,CNC(=O)C[C@H]1COc2cc(F)c(CC(C)C)cc2N1C(=O)c1cc...,-1.287433,-1.041104,-2.580540,-2.580540,-1.690871,-1.151363,-1.082584,-1.556482,-3.004700,...,-1.946023,-2.126862,-1.842472,-1.203783,-1.059797,-1.059803,-1.227556,-1.314220,-0.884677,-1.952187
2,CNC(=O)c1cccc(CC[C@]2(O)CCC3=Cc4c(cnn4-c4ccc(F...,-1.196572,-1.640617,-2.300717,-2.300717,-2.340469,-2.176364,-1.582061,-1.918152,-2.644014,...,-1.950960,-2.413268,-0.747447,-0.873330,-0.852091,-0.852097,-0.894819,-0.731963,-0.532396,-1.751119
3,CC(=O)[C@@]1(O)CC[C@H]2[C@@H]3CCC4=CC(=O)CC[C@...,-2.116156,-2.062974,0.388856,0.388855,0.251030,-0.468169,-2.138901,0.307127,0.503793,...,-1.169076,-0.024440,-0.875208,-0.811158,-0.742382,-0.742381,-0.758479,-0.532316,-0.340365,-1.956484
4,Cn1cc(S(=O)(=O)N2CC[C@H]3Cc4c(cnn4-c4ccc(F)cc4...,-1.025681,-0.770819,-1.509483,-1.509483,-1.170218,-0.994044,-0.917058,-0.824099,-1.887813,...,-1.943925,-0.668546,-0.856471,-1.048207,-0.824520,-0.824521,-0.705930,-0.871579,-0.422264,-1.151568
